In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
class DataHandler:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.input_df = None
        self.output_df = None
        self.x_train = self.x_test = self.y_train = self.y_test = None
        self.target_encoder = None

    def load_data(self):
        self.df = pd.read_csv(self.file_path, sep=',')

    def handle_nulls_duplicates(self):
        initial_rows = self.df.shape[0]
        self.df.dropna(inplace=True)
        self.df.drop_duplicates(inplace=True)
        
    def feature_engineer_age(self):
        if 'Age' in self.df.columns and self.df['Age'].dtype == 'object':
            self.df['Age'] = self.df['Age'].astype(str).str.replace(' years', '', regex=False).astype(int)
            
    def create_input_output(self, target_column):
        input_columns = [col for col in self.df.columns if col != target_column]
        self.input_df = self.df[input_columns]

        rs_ord = ['Insufficient_Weight', 'Normal_Weight',
                  'Overweight_Level_I', 'Overweight_Level_II',
                  'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']
        self.target_encoder = OrdinalEncoder(categories=[rs_ord])
        self.output_df = pd.DataFrame(self.target_encoder.fit_transform(self.df[[target_column]]), columns=[target_column])
        
    def split_data(self, test_size=0.2, random_state=354):
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(
            self.input_df, self.output_df, test_size=test_size, random_state=random_state
        )
        return self.x_train, self.x_test, self.y_train, self.y_test

In [3]:
class EncoderHandler:
    def __init__(self):
        self.ordinal_encoders = {}
        self.onehot_encoder = None
        self.encoded_feature_names = None

    def fit_transform(self, df):
        df_encoded = df.copy()

        yn_lab = ['no', 'yes']
        mf_lab = ['Male', 'Female']
        fq_ord = ['no', 'Sometimes', 'Frequently', 'Always']

        ordinal_cols_info = {
            'Gender': mf_lab,
            'FamilyHistory': yn_lab,
            'FAVC': yn_lab,
            'SMOKE': yn_lab,
            'SCC': yn_lab,
            'CAEC': fq_ord,
            'CALC': fq_ord
        }

        for col, categories in ordinal_cols_info.items():
            if col in df_encoded.columns:
                encoder = OrdinalEncoder(categories=[categories])
                df_encoded[col] = encoder.fit_transform(df_encoded[[col]])
                self.ordinal_encoders[col] = encoder

        ohe_column = 'MTRANS'
        if ohe_column in df_encoded.columns:
            self.onehot_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
            enc_arr = self.onehot_encoder.fit_transform(df_encoded[[ohe_column]])
            enc_df = pd.DataFrame(enc_arr, columns=self.onehot_encoder.get_feature_names_out([ohe_column]))

            df_encoded = df_encoded.drop(columns=[ohe_column]).reset_index(drop=True)
            enc_df = enc_df.reset_index(drop=True)
            df_encoded = pd.concat([df_encoded, enc_df], axis=1)

        self.encoded_feature_names = df_encoded.columns.tolist()
        return df_encoded

    def transform(self, df):
        df_encoded = df.copy()

        for col, encoder in self.ordinal_encoders.items():
            if col in df_encoded.columns:
                df_encoded[col] = encoder.transform(df_encoded[[col]])

        ohe_column = 'MTRANS'
        if ohe_column in df_encoded.columns and self.onehot_encoder is not None:
            enc_arr = self.onehot_encoder.transform(df_encoded[[ohe_column]])
            enc_df = pd.DataFrame(enc_arr, columns=self.onehot_encoder.get_feature_names_out([ohe_column]))

            df_encoded = df_encoded.drop(columns=[ohe_column]).reset_index(drop=True)
            enc_df = enc_df.reset_index(drop=True)
            df_encoded = pd.concat([df_encoded, enc_df], axis=1)

        if self.encoded_feature_names:
            missing_cols = set(self.encoded_feature_names) - set(df_encoded.columns)
            for c in missing_cols:
                df_encoded[c] = 0
            df_encoded = df_encoded[self.encoded_feature_names]

        return df_encoded

In [4]:
class ModelHandler:
    def __init__(self, random_state=354):
        self.random_state = random_state
        self.rf_param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [10, 20],
            'min_samples_split': [2],
            'min_samples_leaf': [1],
            'max_features': ['sqrt'],
            'bootstrap': [True],
            'criterion': ['gini']
        }
        self.xg_param_grid = {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [4, 6],
            'subsample': [0.8],
            'colsample_bytree': [0.8],
            'gamma': [0],
            'reg_alpha': [0, 0.1],
            'reg_lambda': [1]
        }

        self.rf_base = RandomForestClassifier(random_state=self.random_state)
        self.xg_base = xgb.XGBClassifier(random_state=self.random_state, tree_method='hist')

        self.rf_grid = GridSearchCV(self.rf_base, self.rf_param_grid, cv=3, n_jobs=-1, scoring='accuracy')
        self.xg_grid = GridSearchCV(self.xg_base, self.xg_param_grid, cv=3, n_jobs=-1, scoring='accuracy')

        self.best_model = None
        self.rf_base_accuracy = 0
        self.xg_base_accuracy = 0
        self.rf_grid_accuracy = 0
        self.xg_grid_accuracy = 0

    def train(self, x_train, y_train):
        self.rf_base.fit(x_train, y_train.values.ravel())
        self.rf_grid.fit(x_train, y_train.values.ravel())

        self.xg_base.fit(x_train, y_train.values.ravel())
        self.xg_grid.fit(x_train, y_train.values.ravel())

    def evaluate(self, x_test, y_test):
        rf_base_preds = self.rf_base.predict(x_test)
        self.rf_base_accuracy = accuracy_score(y_test, rf_base_preds)

        xg_base_preds = self.xg_base.predict(x_test)
        self.xg_base_accuracy = accuracy_score(y_test, xg_base_preds)

        rf_grid_preds = self.rf_grid.predict(x_test)
        self.rf_grid_accuracy = accuracy_score(y_test, rf_grid_preds)

        xg_grid_preds = self.xg_grid.predict(x_test)
        self.xg_grid_accuracy = accuracy_score(y_test, xg_grid_preds)

        print("\n__________BASE RANDOM FOREST__________")
        print("Accuracy: {:.3f}".format(self.rf_base_accuracy))
        print("Classification Report:\n", classification_report(y_test, rf_base_preds, digits=3, zero_division=1))
        print("Confusion Matrix:\n", confusion_matrix(y_test, rf_base_preds))

        print("\n__________BASE XGBOOST__________")
        print("Accuracy: {:.3f}".format(self.xg_base_accuracy))
        print("Classification Report:\n", classification_report(y_test, xg_base_preds, digits=3, zero_division=1))
        print("Confusion Matrix:\n", confusion_matrix(y_test, xg_base_preds))

        print("\n__________TUNED RANDOM FOREST)__________")
        print("Accuracy: {:.3f}".format(self.rf_grid_accuracy))
        print("Best Parameters:", self.rf_grid.best_params_)
        print("Classification Report:\n", classification_report(y_test, rf_grid_preds, digits=3, zero_division=1))
        print("Confusion Matrix:\n", confusion_matrix(y_test, rf_grid_preds))

        print("\n__________TUNED XGBOOST__________")
        print("Accuracy: {:.3f}".format(self.xg_grid_accuracy))
        print("Best Parameters:", self.xg_grid.best_params_)
        print("Classification Report:\n", classification_report(y_test, xg_grid_preds, digits=3, zero_division=1))
        print("Confusion Matrix:\n", confusion_matrix(y_test, xg_grid_preds))

        accuracies = {
            'Base Random Forest': (self.rf_base_accuracy, self.rf_base),
            'Tuned Random Forest': (self.rf_grid_accuracy, self.rf_grid.best_estimator_),
            'Base XGBoost': (self.xg_base_accuracy, self.xg_base),
            'Tuned XGBoost': (self.xg_grid_accuracy, self.xg_grid.best_estimator_)
        }

        best_model_name = max(accuracies, key=lambda k: accuracies[k][0])
        self.best_model = accuracies[best_model_name][1]
        print(f"Best Model: {best_model_name}, accuracy: {accuracies[best_model_name][0]:.5f}")


    def save(self, filename="bestModelFP.pkl"):
        with open(filename, 'wb') as f:
            pickle.dump(self.best_model, f)
        print(f"Final model saved to {filename}")

In [5]:
if __name__ == "__main__":
    file_path = "ObesityDataSet2.csv"
    target_col = "Result"

    data_handler = DataHandler(file_path)
    data_handler.load_data()
    data_handler.handle_nulls_duplicates()
    data_handler.feature_engineer_age()
    data_handler.create_input_output(target_col)
    x_train, x_test, y_train, y_test = data_handler.split_data()

    encoder_handler = EncoderHandler()
    x_train_processed = encoder_handler.fit_transform(x_train)
    x_test_processed = encoder_handler.transform(x_test)

    model_handler = ModelHandler()
    model_handler.train(x_train_processed, y_train)
    model_handler.evaluate(x_test_processed, y_test)

    model_handler.save("fp_fin_mod.pkl")

    with open('fp_ord_enc.pkl', 'wb') as f:
        pickle.dump(encoder_handler.ordinal_encoders, f)

    with open('fp_ohe_enc.pkl', 'wb') as f:
        pickle.dump(encoder_handler.onehot_encoder, f)

    with open('fp_trg_enc.pkl', 'wb') as f:
        pickle.dump(data_handler.target_encoder, f)

    with open('fp_ftr_ord.pkl', 'wb') as f:
        pickle.dump(encoder_handler.encoded_feature_names, f)


__________BASE RANDOM FOREST__________
Accuracy: 0.949
Classification Report:
               precision    recall  f1-score   support

         0.0      1.000     0.955     0.977        22
         1.0      0.759     1.000     0.863        22
         2.0      1.000     0.862     0.926        29
         3.0      0.966     0.933     0.949        30
         4.0      0.941     0.970     0.955        33
         5.0      1.000     0.971     0.985        34
         6.0      1.000     0.963     0.981        27

    accuracy                          0.949       197
   macro avg      0.952     0.950     0.948       197
weighted avg      0.958     0.949     0.951       197

Confusion Matrix:
 [[21  1  0  0  0  0  0]
 [ 0 22  0  0  0  0  0]
 [ 0  3 25  1  0  0  0]
 [ 0  1  0 28  1  0  0]
 [ 0  1  0  0 32  0  0]
 [ 0  0  0  0  1 33  0]
 [ 0  1  0  0  0  0 26]]

__________BASE XGBOOST__________
Accuracy: 0.949
Classification Report:
               precision    recall  f1-score   support

      